### Importación de Librerías

In [1]:
import sys
!{sys.executable} -m pip install openpyxl
import pandas as pd

# Configuración para mostrar todas las columnas
pd.set_option('display.max_columns', None)

### Cargar los datasets desde data/raw

In [2]:
data2023 = pd.read_excel("../data/raw/2023_113_Reporte.xlsx")
data2024 = pd.read_excel("../data/raw/2024_113_Reporte.xlsx")
data2025 = pd.read_excel("../data/raw/2025_113_Reporte.xlsx")

### Agrupar datasets en un diccionario

In [3]:
datasets = {
    "2023": data2023,
    "2024": data2024,
    "2025": data2025
}

### Convertir zscore a número (todos los años)

In [4]:
for año, df in datasets.items():
    if df["zscore_pt"].dtype == object:
        df["zscore_pt"] = df["zscore_pt"].str.replace(",", ".").astype(float)
    print(f"{año}: zscore_pt -> {df["zscore_pt"].dtype}")

2023: zscore_pt -> float64
2024: zscore_pt -> float64
2025: zscore_pt -> float64


### Revisar rangos de edad y validar (todos los años)

In [5]:
for año, df in datasets.items():
    print(f"\n=== {año} ===")
    print(df["edad_"].describe())
    n_mayor5 = (df["edad_"] > 5).sum()
    print(f"Registros con edad > 5: {n_mayor5}")


=== 2023 ===
count    107.000000
mean       2.943925
std        3.012026
min        1.000000
25%        1.000000
50%        2.000000
75%        3.000000
max       13.000000
Name: edad_, dtype: float64
Registros con edad > 5: 16

=== 2024 ===
count    1441.000000
mean        3.222068
std         3.342169
min         1.000000
25%         1.000000
50%         2.000000
75%         4.000000
max        23.000000
Name: edad_, dtype: float64
Registros con edad > 5: 286

=== 2025 ===
count    1057.000000
mean        2.990539
std         3.155368
min         1.000000
25%         1.000000
50%         2.000000
75%         4.000000
max        28.000000
Name: edad_, dtype: float64
Registros con edad > 5: 165


### Ver registros con edad > 5 por año

In [6]:
for año, df in datasets.items():
    resultado = df[df["edad_"] > 5]
    print(f"\n=== {año}: {len(resultado)} registros con edad > 5 ===")
    if len(resultado) > 0:
        print(resultado[["pri_nom_", "pri_ape_", "edad_", "uni_med_"]].to_string())


=== 2023: 16 registros con edad > 5 ===
        pri_nom_   pri_ape_  edad_  uni_med_
0       ELIENDER     IPUANA      6         2
3           DWIA      MEJIA     10         2
8         ROYNER     CAMBAR     11         2
15         PAULA    NAVARRO      8         2
18     MENDERSON     TORRES     10         2
25         JESUS    ALBERTO      9         2
28          JOSE   PUSHAINA     11         2
38        JEIDER     URIANA      8         2
52      DWINAWIN  VILLAFAÑE      6         2
59   HIJO DE ANA  VILLAFAÑE     13         3
72         NAHUM    MONTIEL      9         2
75      ELIENDER     IPUANA     10         2
76      ELIENDER     IPUANA     10         2
90       MARIANA    CAPITAN     11         2
95      ELIENDER     IPUANA     10         2
104      DUKARIN    MARQUEZ      9         2

=== 2024: 286 registros con edad > 5 ===
              pri_nom_    pri_ape_  edad_  uni_med_
2                 YAIR      MONTES     10         2
9                NILVA      URIANA      9       

### Revisar consistencia clas_peso vs zscore (todos los años)

In [7]:
for año, df in datasets.items():
    print(f"\n=== {año} ===")
    print(df.groupby("clas_peso")["zscore_pt"].describe())


=== 2023 ===
           count       mean       std      min        25%       50%  \
clas_peso                                                             
1           14.0  -3.823543  0.672357  -5.2786  -4.040900  -3.69400   
2           66.0  -2.408103  0.250499  -2.9910  -2.526000  -2.37765   
3           11.0  -1.625573  0.221141  -1.9727  -1.799750  -1.63360   
4           10.0  -0.298230  0.590471  -0.9999  -0.754900  -0.46180   
5            3.0   1.734100  0.255253   1.4886   1.602100   1.71560   
6            1.0   2.516400       NaN   2.5164   2.516400   2.51640   
7            2.0  16.726150  0.732916  16.2079  16.467025  16.72615   

                 75%      max  
clas_peso                      
1          -3.333750  -3.0366  
2          -2.207775  -2.0297  
3          -1.423050  -1.2858  
4           0.086600   0.7653  
5           1.856850   1.9981  
6           2.516400   2.5164  
7          16.985275  17.2444  

=== 2024 ===
           count      mean       std     min

### Detectar outliers en peso_act, talla_act e imc (todos los años)

In [8]:
cols = ["peso_act", "talla_act", "imc"]

for año, df in datasets.items():
    for col in cols:
        df[col] = df[col].astype(str).str.replace(",", ".")
        df[col] = pd.to_numeric(df[col], errors="coerce")
    print(f"\n=== {año} ===")
    print(df[cols].describe())


=== 2023 ===
         peso_act   talla_act         imc
count  107.000000  107.000000  107.000000
mean     7.612617   70.724299   15.583084
std      4.560459   10.136240   13.482289
min      2.300000   45.000000   10.000000
25%      5.800000   65.000000   13.100000
50%      7.100000   71.500000   13.700000
75%      8.250000   76.250000   14.135000
max     46.000000   98.500000  123.600000

=== 2024 ===
          peso_act    talla_act          imc
count  1440.000000  1440.000000  1440.000000
mean      7.247083    72.196396    13.702132
std       2.212391     9.998780     3.466012
min       1.600000    45.000000     7.200000
25%       6.100000    66.500000    13.000000
50%       7.100000    72.000000    13.500000
75%       8.300000    77.850000    14.100000
max      34.000000   104.500000   110.900000

=== 2025 ===
          peso_act    talla_act          imc
count  1057.000000  1057.000000  1057.000000
mean      7.479953    72.921760    13.856433
std       2.944117    11.295138     6.84

### Revisar duplicados (todos los años)

In [9]:
for año, df in datasets.items():
    n_dup = df.duplicated().sum()
    print(f"{año}: {n_dup} registros duplicados")

2023: 0 registros duplicados
2024: 0 registros duplicados
2025: 0 registros duplicados


### Revisar valores negativos o cero en peso_act (todos los años)

In [10]:
for año, df in datasets.items():
    df["peso_act"] = df["peso_act"].astype(str).str.replace(",", ".")
    df["peso_act"] = pd.to_numeric(df["peso_act"], errors="coerce")
    n_invalidos = (df["peso_act"] <= 0).sum()
    print(f"{año}: {n_invalidos} registros con peso_act <= 0 | dtype: {df["peso_act"].dtype}")

2023: 0 registros con peso_act <= 0 | dtype: float64
2024: 0 registros con peso_act <= 0 | dtype: float64
2025: 0 registros con peso_act <= 0 | dtype: float64


### Revisar columnas con más valores nulos (todos los años)

In [11]:
for año, df in datasets.items():
    print(f"\n=== {año} ===")
    print(df.isnull().sum().sort_values(ascending=False).head(10))


=== 2023 ===
otra_ident                       107
otra_orien                       107
sem_ges_                         107
fm_grado                         107
cbmte_                           107
cer_def_                         107
fm_unidad                        107
estrato_datos_complementarios    107
RegIniFec                        107
fm_fuerza                        107
dtype: int64

=== 2024 ===
otra_ident                       1441
sem_ges_                         1441
fm_unidad                        1441
fm_grado                         1441
cbmte_                           1441
cer_def_                         1441
RegIniFec                        1441
estrato_datos_complementarios    1441
fm_fuerza                        1441
otra_orien                       1438
dtype: int64

=== 2025 ===
otra_ident                       1057
otra_orien                       1057
sem_ges_                         1057
fm_grado                         1057
cbmte_                        